In [1]:
# First Working Network

In [2]:
# Define the Network


import torch
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # 1 input channel, 6 output channels, 5x5 kernel
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        # fully connected layers
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        # conv -> relu -> max pool
        x = F.max_pool2d(F.relu(self.conv1(x)), (2, 2))
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = torch.flatten(x, 1)  # flatten all dims except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

net = Net()
print(net)


Net(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


In [3]:
# Inspect Learnable Parameters

params = list(net.parameters())
print("Number of parameter tensors:", len(params))
print("conv1 weight shape:", params[0].size())


Number of parameter tensors: 10
conv1 weight shape: torch.Size([6, 1, 5, 5])


In [4]:
# Run a Forward Pass

input_tensor = torch.randn(1, 1, 32, 32)
output = net(input_tensor)
print("Output shape:", output.shape)
print("Output:", output)


Output shape: torch.Size([1, 10])
Output: tensor([[ 0.1152,  0.0557, -0.0058,  0.1089, -0.0289, -0.0212, -0.0974,  0.0709,
          0.1068,  0.0317]], grad_fn=<AddmmBackward0>)


In [5]:
# Compute a Loss Against a Dummy Target

target = torch.randn(10)          # dummy target, same shape as output
target = target.view(1, -1)       # reshape to match output shape

criterion = nn.MSELoss()
loss = criterion(output, target)
print("Loss:", loss.item())


Loss: 1.2737014293670654


In [6]:
# Backpropagate the Loss

net.zero_grad()  # clear existing gradients

print("conv1.bias.grad before backward:", net.conv1.bias.grad)
loss.backward()
print("conv1.bias.grad after backward:", net.conv1.bias.grad)


conv1.bias.grad before backward: None
conv1.bias.grad after backward: tensor([-0.0020, -0.0103,  0.0086, -0.0273, -0.0184,  0.0085])


In [7]:
# One Manual Weight Update (Gradient Descent Step)

learning_rate = 0.01
for f in net.parameters():
    f.data.sub_(f.grad.data * learning_rate)

print("Weights updated.")


Weights updated.
